# 12 — Particle tracking vs. July 2025 drifter deploys

**Goal:** reproduce the 12 Stokes-drifter deploys from the July 2025 field campaign with a Lagrangian particle tracking run driven by the v03a D-Flow FM surface velocity field, then score skill per deploy.

**Data:** `drifters_Jul25_polished.csv` — columns: `Data Date; Sea Surface; x_coord; y_coord; source; deploy; Distance; Time step; Velocity; deltaX; deltaY; quadrante; Direction; Direction N; ...`. Coordinates are UTM Zone 32N (EPSG:32632). Rows with `deploy=altri` are between-deploy drift / anomalies — excluded here.

**Approach (v03a + OpenDrift):**
1. Parse and QC drifter CSV (Italian locale, semicolon separator, comma decimals, mixed date formats)
2. Group by `(deploy, source)` → one drifter track each; first valid row = release (t₀, x₀, y₀)
3. Convert UTM32N → WGS84 to match v03a mesh CRS
4. Run Lagrangian tracking with **OpenDrift** (`OceanDrift` + `reader_netCDF_CF_generic`) over the regridded v03a surface current field
5. Score trajectory skill per deploy: endpoint separation, path-length ratio, Liu & Weisberg skill

**Why regridded input to OpenDrift:** OpenDrift's native readers don't accept D-Flow FM unstructured partitioned output directly (`reader_netCDF_CF_unstructured` requires Cartesian coords, `reader_FVCOM_xarray` needs FVCOM-specific time vars). Simplest path: regrid the 4 MPI-partitioned map.nc onto a rectilinear 0.002° (~200 m) lat-lon grid covering July 7-10 and save as a CF-compliant NetCDF (`data/processed/v03a_surface_current.nc`, 29 MB) via `scripts/regrid_v03a_for_opendrift.py`. OpenDrift then reads it with the standard generic reader.

**OpenDrift advantages over a naive tracker:**
- Well-tested RK4 advection with proper mesh masking
- Stokes drift, windage, horizontal random-walk diffusion configurable
- Coastline bounce-back handling
- Trajectory output in standard NetCDF format

An earlier version of this notebook used a custom RK2 tracker — it has been replaced by the OpenDrift pipeline documented below.

## 1. Imports and paths

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pyproj import Transformer

project_root = Path(r'F:\StagnoneDT')
raw_dir = project_root / 'data' / 'raw' / 'insitu'
processed_dir = project_root / 'data' / 'processed'
fig_dir = project_root / 'figures'
processed_dir.mkdir(parents=True, exist_ok=True)

drifter_csv = raw_dir / 'drifters_Jul25_polished.csv'

# v03a output. IMPORTANT: open_partitioned_dataset needs a WILDCARD pattern
# covering all partitions — passing a single "_0000_map.nc" loads only 1/4 of the domain.
v03a_output = project_root / 'model' / 'dflowfm_v03a' / 'DFM_OUTPUT_Stagnone_dxy01_15m'
v03a_map_pattern = str(v03a_output / 'Stagnone_dxy01_15m_0*_map.nc')

print(f'Drifter CSV exists: {drifter_csv.exists()}  ({drifter_csv})')
print(f'v03a map pattern: {v03a_map_pattern}')
partitions_found = list(v03a_output.glob('Stagnone_dxy01_15m_0*_map.nc'))
print(f'Partitions matching wildcard: {len(partitions_found)}')
for p in sorted(partitions_found):
    print(f'  {p.name}  ({p.stat().st_size / 1e9:.2f} GB)')

## 2. Parse drifter CSV

Handles: UTF-8 BOM, semicolon delimiter, comma decimals, and two date formats present in the file (most rows `MM/DD/YYYY HH:MM`; a handful of QC-flagged rows use `DD/MM/YYYY HH:MM:SS`). Campaign spans **8–9 July 2025**.

Action: load raw, parse dates robustly, drop rows without coords, filter `deploy ∈ {1..12}` (drop `altri`).

In [ ]:
raw = pd.read_csv(
    drifter_csv,
    sep=';',
    encoding='utf-8-sig',
    decimal=',',
    engine='python',
)
# strip trailing empty columns from the messy Excel export
raw = raw.loc[:, ~raw.columns.str.match(r'^Unnamed')]
raw.columns = [c.strip() for c in raw.columns]
print(f'Raw rows: {len(raw)}')
print(f'Columns: {list(raw.columns)}')
raw.head(3)

In [ ]:
def parse_date(s):
    """Parse either MM/DD/YYYY HH:MM or DD/MM/YYYY HH:MM:SS. Returns NaT on failure."""
    if pd.isna(s):
        return pd.NaT
    s = str(s).strip()
    # rows with seconds use DD/MM/YYYY
    if s.count(':') == 2:
        return pd.to_datetime(s, format='%d/%m/%Y %H:%M:%S', errors='coerce')
    # main rows: MM/DD/YYYY HH:MM
    return pd.to_datetime(s, format='%m/%d/%Y %H:%M', errors='coerce')

raw['time'] = raw['Data Date'].apply(parse_date)

# Sanity: campaign was 8-9 July 2025
print('Time range:', raw['time'].min(), '→', raw['time'].max())
print('Rows with unparsed time:', raw['time'].isna().sum())

In [ ]:
# Select and clean
df = raw[['time', 'x_coord', 'y_coord', 'source', 'deploy', 'Velocity [m/s]', 'Direction N [\u00ba]']].copy()
df.columns = ['time', 'x_utm', 'y_utm', 'source', 'deploy', 'v_ms', 'dir_deg']

# Drop rows without time or coords
df = df.dropna(subset=['time', 'x_utm', 'y_utm']).reset_index(drop=True)

# Keep only numbered deploys 1-12 (drop 'altri' and any other non-numeric)
df['deploy'] = pd.to_numeric(df['deploy'], errors='coerce')
df = df.dropna(subset=['deploy'])
df['deploy'] = df['deploy'].astype(int)
df = df[df['deploy'].between(1, 12)].reset_index(drop=True)

print(f'After filtering: {len(df)} rows across deploys {sorted(df.deploy.unique())}')
df.groupby('deploy').agg(
    n_points=('time', 'size'),
    n_drifters=('source', 'nunique'),
    t_start=('time', 'min'),
    t_end=('time', 'max'),
)

## 3. Convert UTM33N → WGS84

**Important:** Drifter coordinates are in **UTM Zone 33N** (EPSG:32633), *not* Zone 32N. Stagnone at 12.45°E lies in Zone 33 (Zone 32 covers 6°E–12°E, Zone 33 covers 12°E–18°E). Using Zone 32 by mistake produces longitudes ~6° off.

Sanity check: transformed coords should land near `lon ~ 12.44–12.49°E, lat ~ 37.84–37.90°N`.

In [ ]:
transformer = Transformer.from_crs('EPSG:32633', 'EPSG:4326', always_xy=True)
df['lon'], df['lat'] = transformer.transform(df['x_utm'].values, df['y_utm'].values)

print(f'Lon range: {df.lon.min():.4f} .. {df.lon.max():.4f}')
print(f'Lat range: {df.lat.min():.4f} .. {df.lat.max():.4f}')
# Expected: lon 12.44–12.49, lat 37.84–37.90 (Stagnone)
assert 12.0 < df.lon.min() and df.lon.max() < 13.0, 'lon out of expected Stagnone range — check CRS!'

## 4. Per-deploy release points

For each `(deploy, source)` we take the first row in time as the release point. These feed the Lagrangian tracker as `t₀, x₀, y₀` seeds.

In [ ]:
df = df.sort_values(['deploy', 'source', 'time']).reset_index(drop=True)
releases = df.groupby(['deploy', 'source'], as_index=False).first()[
    ['deploy', 'source', 'time', 'lon', 'lat']
]
releases.columns = ['deploy', 'drifter_id', 't0', 'lon0', 'lat0']

print(f'Total drifter seeds: {len(releases)}')
releases

In [ ]:
# Save release seeds and cleaned tracks for downstream use
releases_path = processed_dir / 'drifter_releases_Jul2025.csv'
tracks_path = processed_dir / 'drifter_tracks_Jul2025.csv'
releases.to_csv(releases_path, index=False)
df[['deploy', 'source', 'time', 'lon', 'lat', 'x_utm', 'y_utm', 'v_ms', 'dir_deg']].to_csv(tracks_path, index=False)
print(f'Wrote:\n  {releases_path}\n  {tracks_path}')

## 5. Visualize observed tracks

Color by deploy number so spatial clustering is obvious. The campaign had drifters released at various points inside and across the inlets — this is our ground truth for the Lagrangian run.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 11))
cmap = plt.get_cmap('tab20')
for i, (dep, g) in enumerate(df.groupby('deploy')):
    color = cmap(i % 20)
    for _, sg in g.groupby('source'):
        ax.plot(sg['lon'], sg['lat'], '-', color=color, alpha=0.6, lw=1)
    # release points
    seeds = releases[releases['deploy'] == dep]
    ax.scatter(seeds['lon0'], seeds['lat0'], color=color, s=45, marker='o',
               edgecolor='k', linewidth=0.6, label=f'D{dep} (n={len(seeds)})', zorder=5)

ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('Stagnone drifter tracks — July 2025 (12 deploys)')
ax.legend(loc='upper right', fontsize=8, ncol=2)
ax.set_aspect(1 / np.cos(np.radians(37.87)))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Lagrangian tracking with OpenDrift

OpenDrift is more physically complete than the naive RK2 tracker: it handles Stokes drift, windage (direct wind stress on partially-submerged drifters), horizontal random-walk diffusion, and mesh masking properly.

OpenDrift's native readers don't accept D-Flow FM unstructured format directly:
- `reader_netCDF_CF_unstructured` only supports Cartesian coordinates
- `reader_FVCOM_xarray` expects FVCOM-specific time encoding
- `reader_netCDF_CF_generic` works with rectilinear grids

We regridded the 4-partition map.nc surface velocities onto a rectilinear ~0.002° (~200 m) grid covering July 7-10 (3 days around the drifter campaign) via `scripts/regrid_v03a_for_opendrift.py` → `data/processed/v03a_surface_current.nc` (29 MB). Now we can use the generic reader.

Drifter physics in OpenDrift OceanDrift:
- Surface current advection (our regridded field)
- Windage / drift factor (tune via `drift:stokes_drift` or explicit `drift_factor`)
- Horizontal random-walk diffusion

In [ ]:
from datetime import timedelta
from opendrift.readers.reader_netCDF_CF_generic import Reader as GenericReader
from opendrift.models.oceandrift import OceanDrift

# Regridded surface current + wind file
regridded_nc = processed_dir / 'v03a_surface_current.nc'
assert regridded_nc.exists(), f'Missing {regridded_nc} — run scripts/regrid_v03a_for_opendrift.py first'
print(f'Using regridded file: {regridded_nc} ({regridded_nc.stat().st_size / 1e6:.1f} MB)')

reader = GenericReader(str(regridded_nc))
print(f'Reader time: {reader.start_time} .. {reader.end_time}')
print(f'Reader domain: lon [{reader.xmin:.3f}, {reader.xmax:.3f}], lat [{reader.ymin:.3f}, {reader.ymax:.3f}]')
print(f'Variables: {reader.variables}')

o = OceanDrift(loglevel=30)
o.add_reader(reader)

# Drift configuration (wind drift factor is PER-ELEMENT, set via seed_elements, not config)
o.set_config('drift:horizontal_diffusivity', 0.1)
o.set_config('drift:advection_scheme', 'runge-kutta4')
o.set_config('general:coastline_action', 'previous')

WIND_DRIFT_FACTOR = 0.02   # Stokes drifter windage: 1% (deep) to 4% (shallow). 2% is reasonable first guess.
print(f'\nWind drift factor per particle: {WIND_DRIFT_FACTOR} ({WIND_DRIFT_FACTOR*100:.1f}% of wind)')

releases_valid = releases[
    (pd.to_datetime(releases['t0']) >= pd.Timestamp(reader.start_time)) &
    (pd.to_datetime(releases['t0']) <= pd.Timestamp(reader.end_time) - pd.Timedelta(hours=1))
].reset_index(drop=True)
print(f'Valid releases: {len(releases_valid)} / {len(releases)}')

for _, row in releases_valid.iterrows():
    o.seed_elements(
        lon=row['lon0'], lat=row['lat0'],
        time=pd.to_datetime(row['t0']).to_pydatetime(),
        number=1, z=0,
        wind_drift_factor=WIND_DRIFT_FACTOR,   # per-element windage
    )
print(f'Seeded {o.num_elements_scheduled()} particles')

run_duration = timedelta(hours=6)
run_end = min(
    pd.Timestamp(reader.end_time).to_pydatetime(),
    pd.to_datetime(releases_valid['t0'].max()).to_pydatetime() + run_duration,
)
out_file = str(processed_dir / 'opendrift_v03a.nc')
print(f'\nRunning OpenDrift until {run_end}...')
o.run(end_time=run_end, time_step=300, time_step_output=600, outfile=out_file)
print(f'\nDone — {o.num_elements_active()} active, {o.num_elements_deactivated()} deactivated')

In [ ]:
# Read OpenDrift output and reshape to per-drifter DataFrame
# Output file format: (trajectory, time) with lon, lat, status
od_out = xr.open_dataset(out_file)
print(f'OpenDrift output: {dict(od_out.sizes)}')
print(f'Variables: {list(od_out.data_vars)}')

# Build per-drifter track DataFrame
sim_tracks_od = []
for traj_idx in range(od_out.sizes['trajectory']):
    lons = od_out.lon.isel(trajectory=traj_idx).values
    lats = od_out.lat.isel(trajectory=traj_idx).values
    ts = pd.to_datetime(od_out.time.values)
    valid = ~np.isnan(lons) & ~np.isnan(lats)
    if valid.sum() < 2:
        continue
    # Tie this trajectory back to the release row (they were seeded in releases_valid order)
    row = releases_valid.iloc[traj_idx]
    arr = pd.DataFrame({
        'time': ts[valid],
        'lon': lons[valid],
        'lat': lats[valid],
        'deploy': row['deploy'],
        'drifter_id': row['drifter_id'],
    })
    sim_tracks_od.append(arr)

sim_df = pd.concat(sim_tracks_od, ignore_index=True) if sim_tracks_od else pd.DataFrame()
print(f'\nSimulated {len(sim_tracks_od)} trajectories ({len(sim_df)} total positions)')
sim_df.to_csv(processed_dir / 'drifter_sim_v03a.csv', index=False)
print(f'Saved drifter_sim_v03a.csv')

## 7. Skill metrics (per deploy)

When the simulated trajectory file is available, compute per-drifter:
- **Endpoint separation:** great-circle distance between observed and simulated final positions.
- **Path length ratio:** sim_pathlen / obs_pathlen (1 = perfect).
- **Mean heading bias:** circular mean of (sim_heading − obs_heading) along the track.
- **Liu & Weisberg (2011) skill score** `s = 1 − 〈d〉/〈L〉` where d is separation and L is cumulative observed path — the standard trajectory metric in oceanography.

In [ ]:
def haversine_m(lon1, lat1, lon2, lat2):
    R = 6371000.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def liu_weisberg_skill(obs_lon, obs_lat, sim_lon, sim_lat):
    """Liu & Weisberg (2011) trajectory skill. Arrays must be at matching times."""
    d = haversine_m(obs_lon, obs_lat, sim_lon, sim_lat)
    step = haversine_m(obs_lon[:-1], obs_lat[:-1], obs_lon[1:], obs_lat[1:])
    L = np.cumsum(np.concatenate([[0.0], step]))
    valid = L > 0
    if not valid.any():
        return np.nan
    c = (d[valid] / L[valid]).sum() / valid.sum()
    return max(0.0, 1.0 - c)

# --- Score each drifter ---
metrics = []
for (dep, drift_id), sim in sim_df.groupby(['deploy', 'drifter_id']):
    sim = sim.sort_values('time').reset_index(drop=True)
    # Get observed track for this drifter
    obs = df[(df['deploy'] == dep) & (df['source'] == drift_id)].sort_values('time').reset_index(drop=True)
    if len(obs) < 3:
        continue
    # Interpolate obs to sim times (linear in lon/lat)
    obs_t = obs['time'].values.astype('datetime64[s]').astype(float)
    sim_t = sim['time'].values.astype('datetime64[s]').astype(float)
    # Keep only sim steps within obs time range
    mask = (sim_t >= obs_t.min()) & (sim_t <= obs_t.max())
    if mask.sum() < 3:
        continue
    st = sim_t[mask]
    obs_lon = np.interp(st, obs_t, obs['lon'].values)
    obs_lat = np.interp(st, obs_t, obs['lat'].values)
    sim_lon = sim['lon'].values[mask]
    sim_lat = sim['lat'].values[mask]

    # Endpoint separation (final position)
    endpoint_sep = haversine_m(obs_lon[-1], obs_lat[-1], sim_lon[-1], sim_lat[-1])
    # Path lengths
    obs_path = haversine_m(obs_lon[:-1], obs_lat[:-1], obs_lon[1:], obs_lat[1:]).sum()
    sim_path = haversine_m(sim_lon[:-1], sim_lat[:-1], sim_lon[1:], sim_lat[1:]).sum()
    # L&W skill
    skill = liu_weisberg_skill(obs_lon, obs_lat, sim_lon, sim_lat)

    metrics.append({
        'deploy': dep, 'drifter_id': drift_id,
        'n_steps': mask.sum(),
        'endpoint_sep_m': endpoint_sep,
        'obs_path_m': obs_path, 'sim_path_m': sim_path,
        'path_ratio': sim_path / obs_path if obs_path > 0 else np.nan,
        'LW_skill': skill,
    })

metrics_df = pd.DataFrame(metrics)
print(f'Scored {len(metrics_df)} drifter tracks')
if len(metrics_df):
    print('\n=== Per-drifter metrics ===')
    print(metrics_df.sort_values(['deploy', 'drifter_id']).round(3).to_string(index=False))
    print(f'\nMean endpoint separation: {metrics_df["endpoint_sep_m"].mean():.0f} m')
    print(f'Mean path ratio:          {metrics_df["path_ratio"].mean():.3f}')
    print(f'Mean L&W skill:           {metrics_df["LW_skill"].mean():.3f}')
    metrics_df.to_csv(processed_dir / 'drifter_metrics_v03a.csv', index=False)
    print(f'Saved drifter_metrics_v03a.csv')

In [ ]:
# --- Visual comparison: observed vs simulated tracks, with Stagnone landboundary ---

def parse_ldb(path):
    """Parse a Delft3D .ldb landboundary file into a list of (lon, lat) polylines.
    Format per block: name line, 'npts ncols' line, then npts coordinate rows."""
    polylines = []
    with open(path) as f:
        lines = [ln.strip() for ln in f if ln.strip() and not ln.startswith('*')]
    i = 0
    while i < len(lines):
        i += 1  # skip block name
        if i >= len(lines):
            break
        npts = int(lines[i].split()[0])
        i += 1
        coords = np.array([list(map(float, lines[i+k].split()[:2])) for k in range(npts)])
        polylines.append(coords)
        i += npts
    return polylines

ldb_path = project_root / 'model' / 'dflowfm_v03a' / 'Stagnone_dxy01_15m.ldb'
landboundary = parse_ldb(ldb_path) if ldb_path.exists() else []

fig, ax = plt.subplots(figsize=(11, 12))

# Landboundary (grey polygon for coastline context)
for poly in landboundary:
    ax.fill(poly[:, 0], poly[:, 1], color='lightgrey', alpha=0.6, zorder=0)
    ax.plot(poly[:, 0], poly[:, 1], '-', color='grey', lw=0.8, zorder=1)

cmap = plt.get_cmap('tab20')
deploys = sorted(sim_df['deploy'].unique())
for i, dep in enumerate(deploys):
    color = cmap(i % 20)
    # Observed (all drifters in deploy)
    obs_dep = df[df['deploy'] == dep]
    for _, sg in obs_dep.groupby('source'):
        ax.plot(sg['lon'], sg['lat'], '-', color=color, alpha=0.45, lw=1.0)
    # Simulated
    sim_dep = sim_df[sim_df['deploy'] == dep]
    for _, ss in sim_dep.groupby('drifter_id'):
        ax.plot(ss['lon'], ss['lat'], '--', color=color, alpha=0.95, lw=1.2)
    # Release point
    seeds = releases[releases['deploy'] == dep]
    ax.scatter(seeds['lon0'], seeds['lat0'], color=color, s=60, marker='o',
               edgecolor='k', linewidth=0.7, label=f'D{dep}', zorder=5)

ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.set_title('Drifter tracks: observed (solid) vs v03a simulated (dashed)\nOpenDrift + 2% windage; Stagnone landboundary in grey')
ax.legend(loc='upper right', fontsize=8, ncol=2)
ax.set_aspect(1 / np.cos(np.radians(37.87)))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(fig_dir / 'v03a_drifter_comparison.png', dpi=150)
plt.show()
print(f'Parsed {len(landboundary)} landboundary polyline(s) from {ldb_path.name}')
print(f'Saved {fig_dir}/v03a_drifter_comparison.png')


## 8. Interpretation + next steps

**Expected behavior:**
- Campaign was wind-driven during July 8-9 with surface drift dominated by wind stress on the drifters + surface current
- v03a has Stokes drift coming from SWAN waves, but drifters are mostly wind-surface driven so results will depend on Charnock wind drag
- Endpoint separations of 50-500 m over 6 hours are typical; >1 km indicates systematic surface-drift underestimation
- L&W skill > 0.5 is good; > 0.7 excellent

**If systematic under-drift (paths shorter than observed, low skill):**
- v03a wind drag (`icdtyp=4` Charnock) may be too weak — try fixed `icdtyp=1` with higher coefficient
- SWAN BC is currently constant — real wave-induced Stokes drift missing
- Stokes drifter shape has a drag coefficient ~2-3% of wind: surface current alone won't match

**If systematic over-drift (paths too long):**
- Horizontal diffusivity may need damping
- Check wind file: is blended wind at the right intensity?

**Tuning knobs:**
1. Add a wind-drift component: `(u_drift, v_drift) = (u_model, v_model) + f_wind * (u10, v10)` with `f_wind ≈ 0.03` (3% of wind speed), typical for surface drifters
2. Adjust `TRACK_DURATION_H` — current 6 h covers only the short-range validation
3. Add horizontal diffusivity: random-walk term `± sqrt(2 K dt)` with `K ≈ 0.1-1 m²/s`

**Next paper milestone (Paper 1 - 3D hydrodynamics):**
Run particle tracking on a refined v04 once the hypersaline salinity is fixed and with varied wind drag coefficients. Report skill scores in a table for ~6-8 representative drifters.